In [11]:
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim
import ssl
import time
import base64
import logging
import os
from datetime import datetime

CR_num ="C101266524"


### LOGGING settings ###
log_file = os.path.join('/home/jzwxfg/Documents/notebook-python-env/scriptFiles/', CR_num+'_'+ datetime.now().strftime('%Y%m%d') + '.log')
logger = logging.getLogger()
logger.setLevel(logging.INFO)
# Log Time format
time_format = '%Y-%m-%d %H:%M:%S'
# Every output written on the log file
file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt=time_format)
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

def connect_to_vcenter(host, user, pwd):
    context = ssl._create_unverified_context()
    si = SmartConnect(host=host, user=user, pwd=pwd, sslContext=context)
    return si

def get_vm_by_name(si, vm_name):
    logger.info('Searching %s in Vcenter.',str(vm_name))
    content = si.RetrieveContent()
    obj_view = content.viewManager.CreateContainerView(content.rootFolder, [vim.VirtualMachine], True)
    vm_list = obj_view.view
    obj_view.Destroy()
    for vm in vm_list:
        if vm.name == vm_name:
            logger.info('%s is found in Vcenter.',str(vm_name))
            return vm
    return None

def wait_for_task(task):
    while task.info.state not in [vim.TaskInfo.State.success, vim.TaskInfo.State.error]:
        time.sleep(1)
    if task.info.state == vim.TaskInfo.State.error:
        #print(f"Task failed: {task.info.error}")
        logger.error('Task failed: %s',str(task.info.error))
        raise task.info.error
    return task.info.result

def get_vm_status(vm):
    summary = vm.summary
    config = summary.config
    #print(f"VM Name    : {vm.name}")
    #print(f"VM Status  : {summary.runtime.powerState}")
    logger.info('%s status:%s',str(vm.name),str(summary.runtime.powerState))

def power_off_vm(vm):
    logger.info('%s is shutting down',str(vm))
    if vm.runtime.powerState == vim.VirtualMachinePowerState.poweredOn:
        task = vm.PowerOffVM_Task()
        wait_for_task(task)
        


def process_vm(vm_name, si):
    vm = get_vm_by_name(si, vm_name)
    if vm is None:
        #print(f"VM {vm_name} not found.")
        logger.info('%s is NOT found in Vcenter.',str(vm_name))
        return

    # Status of VM
    get_vm_status(vm)

    # Power Off VM


def main():
    vcenter_host = "dcmipvmvcs033.nam.corp.gm.com"
    vcenter_user = base64.b64decode('YXNfanp3eGZnQG5hbS5jb3JwLmdtLmNvbQ==').decode()
    vcenter_password = base64.b64decode('VXduSExwTmZCb2tKMlBHQw==').decode()
    
    # Server List on VM
    vm_names = [ "dcmipavpgs036","TestSrv01","TestSrv02"]  

    logger.info('%s is starting',str(CR_num))
    si = connect_to_vcenter(vcenter_host, vcenter_user, vcenter_password)
    logger.info('Connected to Vcenter')
    try:
        for vm_name in vm_names:
            process_vm(vm_name, si)
            #print("="*50)
        logger.info('%s is completed',str(CR_num))
    
    finally:
        Disconnect(si)
        logger.info('Disconnected from Vcenter')

if __name__ == "__main__":
    main()
